<a href="https://colab.research.google.com/github/345bc/th-deep-learning/blob/main/bt_tuan04_RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#RECURRENT NEURAL NETWORK

### 1.  Nap các thư viện cần thiết

In [1]:
from pandas import read_csv
import numpy as np
import math
import matplotlib.pyplot as plt
from keras.models import Sequential
from keras.layers import Dense, SimpleRNN
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

Tạo hàm định nghĩa cấu trúc cấu trúc RNN

In [2]:
def create_RNN(hidden_units, dense_units, input_shape, activation):
    """
    hidden_units: Số lượng unit trong tầng ẩn RNN (ví dụ: 2 hoặc 3)
    dense_units: Số lượng unit đầu ra của tầng Dense (thường là 1 cho bài toán dự báo đơn biến)
    input_shape: Định dạng đầu vào dưới dạng (time_steps, features)
    activation: Danh sách hàm kích hoạt cho tầng RNN và tầng Dense, ví dụ: ['linear', 'linear'] hoặc ['tanh', 'tanh']
    """
    model = Sequential()

    # Thêm tầng SimpleRNN
    model.add(SimpleRNN(hidden_units, input_shape=input_shape, activation=activation[0]))

    # Thêm tầng Feedforward đầu ra
    model.add(Dense(dense_units, activation=activation[1]))

    # Biên dịch mô hình với thuật toán tối ưu Adam và hàm mất mát MSE
    model.compile(optimizer='adam', loss='mean_squared_error')

    return model

3. Chuẩn bị và biến đổi dữ liệu

In [3]:
def get_train_test(url, split_percent=0.8):
    # Đọc dữ liệu từ file CSV
    df = read_csv(url, usecols=[1], engine='python')
    data = np.array(df.values.astype('float32'))

    # Chuẩn hóa dữ liệu về khoảng 0 đến 1
    scaler = MinMaxScaler(feature_range=(0, 1))
    data = scaler.fit_transform(data).flatten()

    # Chia dữ liệu thành 2 tập train và test theo tỷ lệ (mặc định là 80/20)
    n = len(data)
    split = int(n * split_percent)
    train_data = data[range(split)]
    test_data = data[split:]

    return train_data, test_data, data

In [4]:
def get_XY(dat, time_steps):
    # Tạo chỉ mục cho mảng mục tiêu Y
    y_ind = np.arange(time_steps, len(dat), time_steps)
    Y = dat[y_ind]

    # Chuẩn bị ma trận các bước thời gian X
    row_x = len(Y)
    X = dat[range(time_steps * row_x)]

    # Định hình lại (Reshape) ma trận X theo yêu cầu của Keras RNN
    X = np.reshape(X, (row_x, time_steps, 1))

    return X, Y

Huấn luyện mô hình

In [5]:
# 1. Tải và xử lý tập dữ liệu mẫu (ví dụ: Sunspots)
sunspots_url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/monthly-sunspots.csv'
train_data, test_data, data = get_train_test(sunspots_url)

# 2. Định cấu hình số bước thời gian là 12 (dự báo dựa trên dữ liệu 12 tháng trước)
time_steps = 12
trainX, trainY = get_XY(train_data, time_steps)
testX, testY = get_XY(test_data, time_steps)

# 3. Tạo mô hình RNN cụ thể với 3 hidden units, hàm kích hoạt 'tanh'
model = create_RNN(hidden_units=3, dense_units=1, input_shape=(time_steps, 1), activation=['tanh', 'tanh'])

# 4. Huấn luyện mô hình với số lượng Epochs (chu kỳ học) là 20
model.fit(trainX, trainY, epochs=20, verbose=1)

Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.0529
Epoch 2/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0438 
Epoch 3/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0394 
Epoch 4/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0375 
Epoch 5/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0353 
Epoch 6/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0329 
Epoch 7/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0307 
Epoch 8/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0288 
Epoch 9/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0272 
Epoch 10/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0259 
Epoch 11/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0248
Epoch 12/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.0236
Epoch 13/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0229
Epoch 14/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.0222
Epoch 15/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.0216
Epoch 16/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/s

5. Đánh giá và kiểm tra kết quả dự báo

In [6]:
def print_error(trainY, testY, train_predict, test_predict):
    # Tính toán sai số RMSE
    train_rmse = math.sqrt(mean_squared_error(trainY, train_predict))
    test_rmse = math.sqrt(mean_squared_error(testY, test_predict))

    # In kết quả
    print('Train RMSE: %.3f RMSE' % (train_rmse))
    print('Test RMSE: %.3f RMSE' % (test_rmse))

# Tiến hành dự báo dựa trên mô hình đã học
train_predict = model.predict(trainX)
test_predict = model.predict(testX)

# Gọi hàm hiển thị sai số
print_error(trainY, testY, train_predict, test_predict)

6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 116ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step
Train RMSE: 0.138 RMSE
Test RMSE: 0.253 RMSE
